# Configuração do Ambiente XAdapt-Drift

Este notebook demonstra como usar a biblioteca XAdapt-Drift para análise de drift em modelos de Machine Learning.

## Configuração do PYTHONPATH

Primeiro, vamos adicionar o diretório raiz da biblioteca ao PYTHONPATH para permitir imports diretos.

In [1]:
# Configurar PYTHONPATH para importar a biblioteca XAdapt-Drift
import sys
import os
from pathlib import Path
import json
import logging
import time
from datetime import datetime
from typing import Dict, Any, Tuple, Optional, List, Union
import shap

# Importando o método de Permutation Importance
from sklearn.inspection import permutation_importance

# Imports para criar um modelo de exemplo
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy import stats
from scipy.stats import wasserstein_distance
from scipy.spatial.distance import jensenshannon
from sklearn.linear_model import LogisticRegression


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score
import sys
plt.style.use('seaborn-v0_8-pastel')
sns.set_palette('pastel')

# Configuração básica de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Determinar o diretório raiz do projeto (assumindo que este notebook está em /examples)
project_root = Path.cwd().parent
print(f"Diretório do projeto: {project_root}")

# Adicionar ao PYTHONPATH se ainda não estiver
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f"✅ Adicionado {project_root} ao PYTHONPATH")
else:
    print("✅ Diretório do projeto já está no PYTHONPATH")

from dataset_analyser import DatasetAnalyzer
from drift_metrics_calculator import DriftMetricsCalculator
from drift_report_generator import DriftReportGenerator
from drift_strategy_adapt import AdaptationStrategyEngine

/home/alexandress/miniconda3/envs/tcc/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Diretório do projeto: /home/alexandress/Documents/tcc/XDrift-Analyzer
✅ Adicionado /home/alexandress/Documents/tcc/XDrift-Analyzer ao PYTHONPATH


# Geração de Dados Sintéticos (6 tipos de colunas)

Implementamos abaixo uma classe simples e configurável que gera:

- Float (normal, lognormal, uniforme)
- Int (Poisson, binomial, uniforme)
- Categórica Numérica (valores discretos numéricos tratados como categorias)
- Categórica String (labels)
- Ordinal (categorias ordenadas)
- Boolean (0/1 ou True/False)

A geração evita complexidade excessiva mas garante variedade suficiente para testes de detecção de drift.


In [2]:
from dataclasses import dataclass, field
from enum import Enum
import numpy as np
import pandas as pd
from typing import List, Dict, Optional, Any

class SimpleDataType(Enum):
    FLOAT = "float"
    INT = "int"
    CAT_NUM = "cat_num"
    CAT_STR = "cat_str"
    ORDINAL = "ordinal"
    BOOLEAN = "bool"

@dataclass
class FeatureSpec:
    name: str
    data_type: SimpleDataType
    n_categories: int = 4
    dist: str = "normal"  # para numéricos: normal|uniform|lognormal|poisson|binomial
    params: Dict[str, Any] = field(default_factory=dict)
    order: Optional[List[str]] = None  # para ordinal

class SimpleSyntheticDataGenerator:
    def __init__(self, random_state: int = 42):
        self.rng = np.random.default_rng(random_state)

    def _gen_float(self, n: int, spec: FeatureSpec):
        dist = spec.dist
        p = {"mean":0, "std":1, "low":0, "high":1, "sigma":0.5} | spec.params
        if dist == "normal":
            data = self.rng.normal(p["mean"], p["std"], n)
        elif dist == "uniform":
            data = self.rng.uniform(p["low"], p["high"], n)
        elif dist == "lognormal":
            data = self.rng.lognormal(p["mean"], p["sigma"], n)
        else:
            data = self.rng.normal(0,1,n)
        return data.astype(float)

    def _gen_int(self, n: int, spec: FeatureSpec):
        dist = spec.dist
        p = {"lam":5, "n":10, "p":0.5, "low":0, "high":100} | spec.params
        if dist == "poisson":
            data = self.rng.poisson(p["lam"], n)
        elif dist == "binomial":
            data = self.rng.binomial(p["n"], p["p"], n)
        elif dist == "uniform":
            data = self.rng.integers(p["low"], p["high"], n)
        else:
            data = self.rng.integers(0, 50, n)
        return data.astype(int)

    def _gen_cat_num(self, n: int, spec: FeatureSpec):
        values = list(range(spec.n_categories))
        probs = self._random_probs(len(values))
        return pd.Categorical(self.rng.choice(values, size=n, p=probs))

    def _gen_cat_str(self, n: int, spec: FeatureSpec):
        values = [f"C{idx}" for idx in range(spec.n_categories)]
        probs = self._random_probs(len(values))
        return pd.Categorical(self.rng.choice(values, size=n, p=probs))

    def _gen_ordinal(self, n: int, spec: FeatureSpec):
        if spec.order is None:
            spec.order = [f"L{i}" for i in range(spec.n_categories)]
        probs = self._random_probs(len(spec.order))
        cat = self.rng.choice(spec.order, size=n, p=probs)
        return pd.Categorical(cat, categories=spec.order, ordered=True)

    def _gen_boolean(self, n: int, spec: FeatureSpec):
        probs = spec.params.get("probs", [0.5, 0.5])
        if abs(sum(probs) - 1.0) > 1e-6:
            probs = [0.5, 0.5]
        return self.rng.choice([0,1], size=n, p=probs).astype(int)

    def _random_probs(self, k: int):
        raw = self.rng.random(k)
        return raw / raw.sum()

    def generate(self, n_samples: int, specs: List[FeatureSpec]):
        data = {}
        for spec in specs:
            if spec.data_type == SimpleDataType.FLOAT:
                data[spec.name] = self._gen_float(n_samples, spec)
            elif spec.data_type == SimpleDataType.INT:
                data[spec.name] = self._gen_int(n_samples, spec)
            elif spec.data_type == SimpleDataType.CAT_NUM:
                data[spec.name] = self._gen_cat_num(n_samples, spec)
            elif spec.data_type == SimpleDataType.CAT_STR:
                data[spec.name] = self._gen_cat_str(n_samples, spec)
            elif spec.data_type == SimpleDataType.ORDINAL:
                data[spec.name] = self._gen_ordinal(n_samples, spec)
            elif spec.data_type == SimpleDataType.BOOLEAN:
                data[spec.name] = self._gen_boolean(n_samples, spec)
        df = pd.DataFrame(data)
        return df


### 🌪️ Indução de Drift (15+ tipos com calibração científica)

Implementação de `SimpleDriftInducer` com tipos calibrados para diferentes magnitudes de efeito:

**Tipos de Shift (mudança de média):**
- `trivial_shift`: severity × 0.1 → D ≈ 0.02-0.04 (LOW)
- `small_shift`: severity × 0.25 → D ≈ 0.05-0.08 (MEDIUM)
- `moderate_shift`: severity × 0.5 → D ≈ 0.10-0.15 (HIGH)
- `large_shift`: severity × 1.0 → D ≈ 0.20+ (CRITICAL)
- `gradual_mean_shift`: mudança linear crescente
- `sudden_mean_shift`: mudança abrupta na metade da amostra

**Tipos de Variância:**
- `variance_increase/decrease`: altera dispersão mantendo média

**Outros Tipos:**
- `seasonal`: padrão senoidal
- `outliers`: injeção de valores extremos
- `new_categories`: adiciona categorias (apenas categóricas)
- `category_frequency_shift`: altera proporções categóricas
- `noise`: ruído gaussiano
- `multiplicative_scale`: escalonamento multiplicativo
- `missingness`: introdução de valores NA
- `distribution_change`: troca normal ↔ lognormal

**Referência:** Thresholds baseados em Cohen (1988) e Sawilowsky (2009) para effect sizes.

In [3]:
class SimpleDriftInducer:
    def __init__(self, random_state: int = 42):
        self.rng = np.random.default_rng(random_state)

    def induce(self, df: pd.DataFrame, config: Dict[str, Dict[str, Any]]):
        drifted = df.copy(deep=True)
        metadata = {}
        for col, spec in config.items():
            drift_type = spec.get("type")
            severity = spec.get("severity", 1.0)
            if drift_type is None or col not in drifted.columns:
                continue
            # Shift types (calibrados para effect sizes específicos)
            if drift_type == "trivial_shift":
                drifted[col], meta = self._sudden_mean_shift(drifted[col], severity*0.1)
                meta['expected_effect'] = 'trivial (D < 0.05)'
            elif drift_type == "small_shift":
                drifted[col], meta = self._sudden_mean_shift(drifted[col], severity*0.25)
                meta['expected_effect'] = 'small (0.05 ≤ D < 0.10)'
            elif drift_type == "moderate_shift":
                drifted[col], meta = self._sudden_mean_shift(drifted[col], severity*0.5)
                meta['expected_effect'] = 'moderate (0.10 ≤ D < 0.20)'
            elif drift_type == "large_shift":
                drifted[col], meta = self._sudden_mean_shift(drifted[col], severity*1.0)
                meta['expected_effect'] = 'large (D ≥ 0.20)'
            # Legacy aliases (mantidos para compatibilidade)
            elif drift_type == "soft_shift":
                drifted[col], meta = self._sudden_mean_shift(drifted[col], severity*0.1)
                meta['warning'] = 'soft_shift deprecated: use trivial_shift instead'
            elif drift_type == "hard_shift":
                drifted[col], meta = self._sudden_mean_shift(drifted[col], severity*1.5)
                meta['warning'] = 'hard_shift deprecated: use large_shift instead'
            # Original types
            elif drift_type == "gradual_mean_shift":
                drifted[col], meta = self._gradual_mean_shift(drifted[col], severity)
            elif drift_type == "sudden_mean_shift":
                drifted[col], meta = self._sudden_mean_shift(drifted[col], severity)
            elif drift_type == "variance_increase":
                drifted[col], meta = self._variance_change(drifted[col], factor=1+0.5*severity)
            elif drift_type == "variance_decrease":
                drifted[col], meta = self._variance_change(drifted[col], factor=max(0.1,1-0.5*severity))
            elif drift_type == "seasonal":
                drifted[col], meta = self._seasonal(drifted[col], severity)
            elif drift_type == "outliers":
                drifted[col], meta = self._outliers(drifted[col], severity)
            elif drift_type == "new_categories":
                drifted[col], meta = self._new_categories(drifted[col], spec.get("n_new",1), severity)
            elif drift_type == "category_frequency_shift":
                drifted[col], meta = self._category_freq_shift(drifted[col], severity)
            elif drift_type == "noise":
                drifted[col], meta = self._noise(drifted[col], severity)
            elif drift_type == "multiplicative_scale":
                drifted[col], meta = self._multiplicative(drifted[col], severity)
            elif drift_type == "missingness":
                drifted[col], meta = self._missingness(drifted[col], severity)
            elif drift_type == "distribution_change":
                drifted[col], meta = self._distribution_change(drifted[col])
            else:
                meta = {"ignored":True}
            metadata[col] = {"drift_type": drift_type, **meta}
        return drifted, metadata

    def _is_numeric(self, s: pd.Series):
        return np.issubdtype(s.dtype, np.number) if not isinstance(s.dtype, pd.CategoricalDtype) else False

    # --- Numeric helpers ---
    def _gradual_mean_shift(self, s: pd.Series, severity: float):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        n = len(s)
        shift = np.linspace(0, severity * (s.std(ddof=0) or 1.0), n)
        return s + shift, {"final_shift": float(shift[-1])}

    def _sudden_mean_shift(self, s: pd.Series, severity: float):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        mid = len(s)//2
        std = s.std(ddof=0) or 1.0
        delta = severity * std
        s2 = s.copy()
        s2.iloc[mid:] = s2.iloc[mid:] + delta
        return s2, {"delta": float(delta)}

    def _variance_change(self, s: pd.Series, factor: float):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        mean = s.mean()
        centered = s - mean
        new = centered * factor + mean
        return new, {"variance_factor": float(factor)}

    def _seasonal(self, s: pd.Series, severity: float):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        n = len(s)
        t = np.arange(n)
        amplitude = severity * (s.std(ddof=0) or 1.0)
        seasonal = amplitude * np.sin(2*np.pi*t/ 50)
        return s + seasonal, {"amplitude": float(amplitude)}

    def _outliers(self, s: pd.Series, severity: float):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        s2 = s.copy()
        n = len(s)
        k = max(1, int(0.01 * severity * n))
        std = s.std(ddof=0) or 1.0
        idx = self.rng.choice(n, size=k, replace=False)
        extreme = s.mean() + 6 * std * (self.rng.choice([-1,1], size=k))
        s2.iloc[idx] = extreme
        return s2, {"n_outliers": int(k)}

    def _noise(self, s: pd.Series, severity: float):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        std = s.std(ddof=0) or 1.0
        scale = severity * 0.2 * std
        return s + self.rng.normal(0, scale, len(s)), {"noise_std": float(scale)}

    def _multiplicative(self, s: pd.Series, severity: float):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        factor = 1 + 0.5 * severity
        return s * factor, {"factor": float(factor)}

    def _missingness(self, s: pd.Series, severity: float):
        s2 = s.copy()
        n = len(s)
        frac = min(0.9, 0.05 * severity)
        k = int(frac * n)
        if k>0:
            idx = self.rng.choice(n, size=k, replace=False)
            s2.iloc[idx] = np.nan
        return s2, {"missing_frac": float(frac)}

    def _distribution_change(self, s: pd.Series):
        if not self._is_numeric(s):
            return s, {"skipped":"non-numeric"}
        std = s.std(ddof=0) or 1.0
        if (s > 0).mean() > 0.9:
            new = (s - s.mean())/ (std+1e-9)
            new = new * std + s.mean()
            return new, {"changed_from":"loglike"}
        else:
            normed = (s - s.mean())/ (std+1e-9)
            new = np.exp(normed)
            return pd.Series(new, index=s.index), {"changed_from":"norm"}

    # --- Categorical helpers ---
    def _new_categories(self, s: pd.Series, n_new: int, severity: float):
        if not isinstance(s.dtype, pd.CategoricalDtype):
            return s, {"skipped":"not-categorical"}
        existing = list(s.cat.categories)
        new_vals = [f"NEW_{i}" for i in range(n_new)]
        s2 = s.copy().cat.add_categories(new_vals)
        n = len(s2)
        frac = min(0.5, 0.1 * severity)
        k = max(1, int(frac * n))
        idx = self.rng.choice(n, size=k, replace=False)
        assign_vals = self.rng.choice(new_vals, size=k)
        s_obj = s2.astype(object)
        for i, v in zip(idx, assign_vals):
            s_obj.iloc[i] = v
        s_final = pd.Categorical(s_obj, categories=existing + new_vals)
        return pd.Series(s_final, index=s.index), {"n_new_categories": n_new, "new_frac": float(k/ n)}

    def _category_freq_shift(self, s: pd.Series, severity: float):
        if not isinstance(s.dtype, pd.CategoricalDtype):
            return s, {"skipped":"not-categorical"}
        s2 = s.copy().astype(object)
        cats = list(s.cat.categories)
        n = len(s)
        target_cat = self.rng.choice(cats)
        frac = min(0.8, 0.2 * severity)
        k = int(frac * n)
        if k>0:
            idx = self.rng.choice(n, size=k, replace=False)
            s2.iloc[idx] = target_cat
        s2 = pd.Categorical(s2, categories=cats)
        return pd.Series(s2, index=s.index), {"target_cat": target_cat, "boost_frac": float(frac)}

In [4]:
# Exemplo rápido de uso
specs = [
    FeatureSpec("f_float0", SimpleDataType.FLOAT, dist="normal", params={"mean":2, "std":1.5}),
    FeatureSpec("f_float1", SimpleDataType.FLOAT, dist="normal", params={"mean":2, "std":1.5}),
    FeatureSpec("f_float2", SimpleDataType.FLOAT, dist="poisson"),
    FeatureSpec("f_int", SimpleDataType.INT, dist="poisson", params={"lam":8}),
    FeatureSpec("f_cat_num", SimpleDataType.CAT_NUM, n_categories=5),
    FeatureSpec("f_cat_str0", SimpleDataType.CAT_STR, n_categories=4),
    FeatureSpec("f_cat_str1", SimpleDataType.CAT_STR, n_categories=5),
    FeatureSpec("f_ord", SimpleDataType.ORDINAL, n_categories=5, order=["A","B","C","D","E"]),
    FeatureSpec("f_bool", SimpleDataType.BOOLEAN, params={"probs":[0.7,0.3]}),
]

simple_gen = SimpleSyntheticDataGenerator(random_state=7)
base_df = simple_gen.generate(1000, specs)
print(base_df.head())
print("Tipos:", base_df.dtypes.to_dict())
print(base_df.describe())


# Recriar demo após ajuste
inducer = SimpleDriftInducer(random_state=10)
drift_config_demo = {
    "f_float0": {"type":"gradual_mean_shift", "severity":1.5},
    # "f_int": {"type":"sudden_mean_shift", "severity":1.0},
    # "f_cat_str0": {"type":"category_frequency_shift", "severity":2.0},
    # "f_cat_num": {"type":"new_categories", "severity":2.0, "n_new":2},
    # "f_bool": {"type":"missingness", "severity":3.0},
    # "f_ord": {"type":"seasonal", "severity":1.0}  # ignorado por ser ordinal (não numérico direto)
}

drifted_df, drift_meta = inducer.induce(base_df, drift_config_demo)
print("Meta:")
for c, m in drift_meta.items():
    print(c, m)
print("\nComparação simples (primeiras linhas):")
print(pd.concat([base_df.head(), drifted_df.head()], axis=1, keys=["orig","drift"]))

   f_float0  f_float1  f_float2  f_int f_cat_num f_cat_str0 f_cat_str1 f_ord  \
0  2.001845  2.538206 -0.365636      8         4         C1         C1     D   
1  2.448118  2.587390  0.341812      8         2         C2         C4     E   
2  1.588793  1.368978 -0.165342     13         2         C0         C0     E   
3  0.664112  5.031334 -0.889232      6         2         C0         C0     A   
4  1.317994  2.556560  0.787681      7         2         C1         C1     D   

   f_bool  
0       0  
1       0  
2       0  
3       0  
4       0  
Tipos: {'f_float0': dtype('float64'), 'f_float1': dtype('float64'), 'f_float2': dtype('float64'), 'f_int': dtype('int64'), 'f_cat_num': CategoricalDtype(categories=[0, 1, 2, 3, 4], ordered=False, categories_dtype=int64), 'f_cat_str0': CategoricalDtype(categories=['C0', 'C1', 'C2', 'C3'], ordered=False, categories_dtype=object), 'f_cat_str1': CategoricalDtype(categories=['C0', 'C1', 'C2', 'C3', 'C4'], ordered=False, categories_dtype=object), 'f

# Persistência de Resultados de Drift & Estratégias

Serão mantidas três tabelas em memória (DataFrames):

1. exec_stats_table (Tabela 1) – Uma linha por execução do pipeline
   Colunas principais:
   - execution_id (int incremental)
   - timestamp
   - total_features
   - features_drifted
   - high_severity_features
   - aggregate_score (score global de drift)
   - strategy_name (estratégia recomendada)
   - strategy_confidence
   - decision (executar, adiar, monitorar, etc. se existir no engine)

2. feature_metrics_table (Tabela 2) – Métricas por feature por execução
   Colunas:
   - execution_id
   - feature
   - metric_name
   - metric_value
   - severity (quando disponível)
   - drift_detected (bool)
   - raw_bucket / interpretation (quando disponível no report)

3. drift_events_table (Tabela 3) – Registro de eventos de drift
   Colunas:
   - event_id
   - execution_id
   - feature
   - event_type (ex: 'drift_detected', 'high_severity')
   - severity
   - strategy_suggested
   - strategy_confidence
   - decision
   - timestamp

A classe abaixo abstrai o processo: `DriftPersistenceManager`.


In [5]:
class DriftPersistenceManager:
    """Gerencia persistência em memória de execuções de detecção de drift e recomendações de adaptação.

    Métodos principais:
      - register_execution(unified_report, strategy_tuple, extra_context=None)
      - get_tables()
      - export(path, format="csv")

    Parâmetros esperados:
      unified_report: dict retornado por DriftReportGenerator.generate_report
      strategy_tuple: (strategy_instance, confidence, decision_dict/opcional)
    """
    def __init__(self):
        # Tabela 1: estatísticas por execução
        self.exec_stats_table = pd.DataFrame(columns=[
            "execution_id","timestamp","total_features","features_drifted",
            "high_severity_features","aggregate_score","strategy_name",
            "strategy_confidence","decision"
        ])
        # Tabela 2: métricas por feature
        self.feature_metrics_table = pd.DataFrame(columns=[
            "execution_id","feature","metric_name","metric_value",
            "severity","drift_detected","interpretation"
        ])
        # Tabela 3: eventos de drift
        self.drift_events_table = pd.DataFrame(columns=[
            "event_id","execution_id","feature","event_type","severity",
            "strategy_suggested","strategy_confidence","decision","timestamp"
        ])
        self._execution_counter = 0
        self._event_counter = 0

    # ---------------- Public API -----------------
    def register_execution(self, unified_report: Dict[str, Any], strategy_tuple: Optional[Tuple[Any, float, Optional[Dict[str, Any]]]] = None, extra_context: Optional[Dict[str, Any]] = None):
        """Registra uma nova execução alimentando as três tabelas.

        strategy_tuple: (strategy_instance, confidence, decision_dict?)
        unified_report estrutura mínima esperada:
          - unified_report['global']['total_features_analyzed']
          - unified_report['global']['features_with_detected_drift'] (ou similar)
          - unified_report['global']['features_with_high_severity']
          - unified_report['global']['aggregate_score']
          - unified_report['features'] = { feature_name: { 'metrics': {...}, 'severity': float, 'drift_detected': bool, 'interpretation': str? } }
        """
        self._execution_counter += 1
        execution_id = self._execution_counter
        ts = datetime.utcnow().isoformat()

        global_section = unified_report.get('global', {})
        features_section = unified_report.get('features', {})

        total_features = global_section.get('total_features_analyzed', len(features_section))
        features_drifted = global_section.get('features_with_detected_drift', global_section.get('features_with_drift', None))
        if features_drifted is None:
            # fallback: contar manually
            features_drifted = sum(1 for f,v in features_section.items() if v.get('drift_detected'))
        high_sev = global_section.get('features_with_high_severity', None)
        if high_sev is None:
            high_sev = sum(1 for f,v in features_section.items() if (v.get('severity') or 0) >= 0.7)
        aggregate_score = global_section.get('aggregate_score', None)

        strategy_name = None
        strategy_conf = None
        decision_value = None
        if strategy_tuple:
            # Tentar descompactar diferentes formatos
            if len(strategy_tuple) == 3:
                strategy_obj, confidence, decision_dict = strategy_tuple
            elif len(strategy_tuple) == 2:
                strategy_obj, confidence = strategy_tuple
                decision_dict = None
            else:
                strategy_obj = strategy_tuple[0]
                confidence = None
                decision_dict = None
            strategy_name = getattr(strategy_obj, 'name', strategy_obj.__class__.__name__ if strategy_obj else None)
            strategy_conf = confidence
            if decision_dict:
                decision_value = decision_dict.get('decision') or decision_dict.get('action') or str(decision_dict)

        # --- Atualiza Tabela 1 ---
        exec_row = {
            'execution_id': execution_id,
            'timestamp': ts,
            'total_features': total_features,
            'features_drifted': features_drifted,
            'high_severity_features': high_sev,
            'aggregate_score': aggregate_score,
            'strategy_name': strategy_name,
            'strategy_confidence': strategy_conf,
            'decision': decision_value
        }
        self.exec_stats_table = pd.concat([self.exec_stats_table, pd.DataFrame([exec_row])], ignore_index=True)

        # --- Atualiza Tabela 2 (métricas por feature) ---
        feature_metric_rows = []
        for feat, info in features_section.items():
            metrics_map = info.get('metrics', {})
            for metric_name, metric_value in metrics_map.items():
                feature_metric_rows.append({
                    'execution_id': execution_id,
                    'feature': feat,
                    'metric_name': metric_name,
                    'metric_value': metric_value,
                    'severity': info.get('severity'),
                    'drift_detected': info.get('drift_detected'),
                    'interpretation': info.get('interpretation') or info.get('category') or None
                })
        if feature_metric_rows:
            self.feature_metrics_table = pd.concat([self.feature_metrics_table, pd.DataFrame(feature_metric_rows)], ignore_index=True)

        # --- Atualiza Tabela 3 (eventos) ---
        event_rows = []
        for feat, info in features_section.items():
            if info.get('drift_detected'):
                self._event_counter += 1
                event_rows.append({
                    'event_id': self._event_counter,
                    'execution_id': execution_id,
                    'feature': feat,
                    'event_type': 'drift_detected',
                    'severity': info.get('severity'),
                    'strategy_suggested': strategy_name,
                    'strategy_confidence': strategy_conf,
                    'decision': decision_value,
                    'timestamp': ts
                })
                # evento adicional se alta severidade
                if (info.get('severity') or 0) >= 0.7:
                    self._event_counter += 1
                    event_rows.append({
                        'event_id': self._event_counter,
                        'execution_id': execution_id,
                        'feature': feat,
                        'event_type': 'high_severity',
                        'severity': info.get('severity'),
                        'strategy_suggested': strategy_name,
                        'strategy_confidence': strategy_conf,
                        'decision': decision_value,
                        'timestamp': ts
                    })
        if event_rows:
            self.drift_events_table = pd.concat([self.drift_events_table, pd.DataFrame(event_rows)], ignore_index=True)

        return execution_id

    def get_tables(self):
        return {
            'exec_stats_table': self.exec_stats_table.copy(),
            'feature_metrics_table': self.feature_metrics_table.copy(),
            'drift_events_table': self.drift_events_table.copy()
        }

    def export(self, path: str, format: str = "csv"):
        path = path.rstrip('/\\')
        if format == 'csv':
            self.exec_stats_table.to_csv(f"{path}/exec_stats.csv", index=False)
            self.feature_metrics_table.to_csv(f"{path}/feature_metrics.csv", index=False)
            self.drift_events_table.to_csv(f"{path}/drift_events.csv", index=False)
        elif format == 'parquet':
            self.exec_stats_table.to_parquet(f"{path}/exec_stats.parquet", index=False)
            self.feature_metrics_table.to_parquet(f"{path}/feature_metrics.parquet", index=False)
            self.drift_events_table.to_parquet(f"{path}/drift_events.parquet", index=False)
        else:
            raise ValueError("Formato não suportado: use 'csv' ou 'parquet'")

    def summary(self):
        if self.exec_stats_table.empty:
            return {"executions":0}
        return {
            "executions": int(self.exec_stats_table['execution_id'].nunique()),
            "total_drift_events": int(self.drift_events_table.shape[0]),
            "features_mostrando_drift_unicos": int(self.drift_events_table['feature'].nunique()) if not self.drift_events_table.empty else 0,
            "estrategias_usadas": self.exec_stats_table['strategy_name'].dropna().unique().tolist()
        }

# Instancia global opcional
persistence_manager = DriftPersistenceManager()
print("DriftPersistenceManager pronto. Use persistence_manager.register_execution(unified_report, (strategy, confidence, decision_dict))")

DriftPersistenceManager pronto. Use persistence_manager.register_execution(unified_report, (strategy, confidence, decision_dict))


In [6]:
# Demo de registro de execução
# Pressupõe que 'unified_report' já exista. Caso contrário, pule ou gere antes.

# try:
#     _ = unified_report  # checa existência
# except NameError:
#     print("unified_report não encontrado - gere um relatório antes de registrar.")
# else:
#     # Mock simples de estratégia caso não tenha sido selecionada ainda
#     class _MockStrategy:
#         name = "MockStrategy"
#     mock_strategy = _MockStrategy()
#     strategy_tuple = (mock_strategy, 0.85, {"decision":"monitor"})

#     exec_id = persistence_manager.register_execution(unified_report, strategy_tuple)
#     tables = persistence_manager.get_tables()
#     print(f"Execução registrada com ID: {exec_id}")
#     print("Resumo:", persistence_manager.summary())
#     print("\nTabela 1 - Exec Stats (head):")
#     display(tables['exec_stats_table'].tail(3))
#     print("\nTabela 2 - Métricas por Feature (head):")
#     display(tables['feature_metrics_table'].head())
#     print("Total linhas métricas:", len(tables['feature_metrics_table']))
#     print("\nTabela 3 - Eventos de Drift (head):")
#     display(tables['drift_events_table'].head())

# Visualizações das Tabelas de Persistência
Serão adicionadas funções utilitárias para gerar gráficos:

1. Evolução do aggregate_score por execução
2. Nº de features com drift vs execuções
3. Distribuição de severidade (histograma ou box) por execução
4. Top N features mais frequentemente com drift
5. Heatmap (execuções x features) presença de drift
6. Linha de tempo de eventos (count por execução) + stacked por tipo
7. Evolução da confiança da estratégia sugerida

Funções lidarão graciosamente com tabelas vazias (early return com aviso).

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

class DriftPlots:
    def __init__(self, persistence_manager: DriftPersistenceManager):
        self.pm = persistence_manager

    def _safe(self, table_name: str):
        tables = self.pm.get_tables()
        df = tables[table_name]
        if df.empty:
            print(f"[AVISO] Tabela '{table_name}' vazia – gere execuções antes de plotar.")
            return None
        return df

    def aggregate_score(self):
        df = self._safe('exec_stats_table')
        if df is None or 'aggregate_score' not in df.columns:
            return
        if df['aggregate_score'].isna().all():
            print("[AVISO] Sem valores de aggregate_score para plotar.")
            return
        plt.figure(figsize=(8,4))
        sns.lineplot(data=df, x='execution_id', y='aggregate_score', marker='o')
        plt.title('Evolução Aggregate Score')
        plt.xlabel('Execução')
        plt.ylabel('Aggregate Score')
        plt.grid(alpha=0.3)
        plt.show()

    def drift_counts(self):
        df = self._safe('exec_stats_table')
        if df is None:
            return
        cols = [c for c in ['features_drifted','high_severity_features'] if c in df.columns]
        if not cols:
            print('[AVISO] Colunas de contagem não encontradas.')
            return
        plt.figure(figsize=(8,4))
        for c in cols:
            sns.lineplot(data=df, x='execution_id', y=c, marker='o', label=c)
        plt.title('Contagem de Features com Drift / Alta Severidade')
        plt.xlabel('Execução')
        plt.ylabel('Quantidade')
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()

    def severity_distribution(self):
        dfm = self._safe('feature_metrics_table')
        if dfm is None:
            return
        if 'severity' not in dfm.columns or dfm['severity'].dropna().empty:
            print('[AVISO] Sem severidades disponíveis.')
            return
        plt.figure(figsize=(8,4))
        sns.boxplot(data=dfm.dropna(subset=['severity']), x='execution_id', y='severity')
        plt.title('Distribuição de Severidade por Execução')
        plt.xlabel('Execução')
        plt.ylabel('Severidade')
        plt.show()

    def top_features_drift(self, top_n=10):
        dfe = self._safe('drift_events_table')
        if dfe is None:
            return
        if dfe.empty:
            print('[AVISO] Sem eventos de drift.')
            return
        counts = (dfe[dfe['event_type']=='drift_detected']
                    .groupby('feature')['event_id']
                    .count()
                    .sort_values(ascending=False)
                    .head(top_n))
        plt.figure(figsize=(8,4))
        sns.barplot(x=counts.values, y=counts.index, orient='h')
        plt.title(f'Top {top_n} Features com Drift Detectado')
        plt.xlabel('Eventos de Drift')
        plt.ylabel('Feature')
        plt.show()

    def drift_heatmap(self):
        dfe = self._safe('drift_events_table')
        if dfe is None or dfe.empty:
            return
        det = dfe[dfe['event_type']=='drift_detected'][['execution_id','feature']]
        if det.empty:
            print('[AVISO] Sem eventos drift_detected para heatmap.')
            return
        det['value'] = 1
        pivot = det.pivot_table(index='feature', columns='execution_id', values='value', aggfunc='sum', fill_value=0)
        plt.figure(figsize=(max(6,0.6*len(pivot.columns)), max(4,0.3*len(pivot.index))))
        sns.heatmap(pivot, annot=True, cmap='Blues', cbar=False, linewidths=.5)
        plt.title('Heatmap de Drift (1 = detectado)')
        plt.xlabel('Execução')
        plt.ylabel('Feature')
        plt.show()

    def events_timeline(self):
        dfe = self._safe('drift_events_table')
        if dfe is None or dfe.empty:
            return
        counts = dfe.groupby(['execution_id','event_type'])['event_id'].count().reset_index()
        plt.figure(figsize=(9,4))
        sns.barplot(data=counts, x='execution_id', y='event_id', hue='event_type')
        plt.title('Eventos por Execução (Stacked)')
        plt.xlabel('Execução')
        plt.ylabel('Eventos')
        plt.legend(title='Tipo')
        plt.show()

    def strategy_confidence(self):
        dfe = self._safe('exec_stats_table')
        if dfe is None:
            return
        if 'strategy_confidence' not in dfe.columns or dfe['strategy_confidence'].dropna().empty:
            print('[AVISO] Sem valores de strategy_confidence.')
            return
        plt.figure(figsize=(8,4))
        sns.lineplot(data=dfe, x='execution_id', y='strategy_confidence', marker='o')
        plt.title('Confiança da Estratégia Sugerida')
        plt.xlabel('Execução')
        plt.ylabel('Confiança')
        plt.ylim(0,1)
        plt.grid(alpha=0.3)
        plt.show()

# Instancia helper
plots = DriftPlots(persistence_manager)
print("Plots helper disponível: plots.aggregate_score(), plots.drift_counts(), plots.severity_distribution(),\nplots.top_features_drift(), plots.drift_heatmap(), plots.events_timeline(), plots.strategy_confidence()")

Plots helper disponível: plots.aggregate_score(), plots.drift_counts(), plots.severity_distribution(),
plots.top_features_drift(), plots.drift_heatmap(), plots.events_timeline(), plots.strategy_confidence()


In [8]:
# # Demo de chamadas de plot (executar após ter múltiplas execuções registradas)
# plots.aggregate_score()
# plots.drift_counts()
# plots.severity_distribution()
# plots.top_features_drift()
# plots.drift_heatmap()
# plots.events_timeline()
# plots.strategy_confidence()

# 🚀 TESTE COMPLETO DA ADAPTATION STRATEGY ENGINE

Agora vamos testar nossa nova **AdaptationStrategyEngine** que utiliza diretamente os relatórios científicos do **DriftReportGenerator** para tomar decisões de adaptação mais inteligentes e baseadas em evidências.

In [9]:
# (Esta célula será substituída pelas novas seções de cenários)

## Cenários Estruturados de Drift com SimpleSyntheticDataGenerator + SimpleDriftInducer

Cada cenário a seguir:
1. Gera dataset base com `SimpleSyntheticDataGenerator`
2. Aplica drift controlado com `SimpleDriftInducer.induce()` usando um `drift_config`
3. Analisa referência vs corrente com `DatasetAnalyzer.analyze_dataset()` (gerando relatório + métricas recomendadas)
4. Calcula métricas de drift com `DriftMetricsCalculator.calculate_metrics_from_report()`
5. Gera relatório consolidado com `DriftReportGenerator` (interpretação científica)
6. Gera recomendação de ação com `AdaptationStrategyEngine` (simulada se necessário)
7. (Opcional) Persiste resultados via `DriftPersistenceManager`

Cenários implementados:
- C1: Mean Shift Suave (gradual_mean_shift)
- C2: Aumento de Variância (variance_increase)
- C3: Alteração de Frequência Categórica (category_frequency_shift)
- C4: Drift Combinado (múltiplos tipos simultâneos)

Cada cenário retornará um dicionário resumido com: severidade global aproximada, nº de métricas calculadas e ação sugerida.


In [10]:
from typing import Dict, Any

# Utilitário principal: executa pipeline completo de um cenário

def run_full_scenario(label: str, generator_specs, n_samples: int, drift_config: Dict[str, Dict[str, Any]],
                      random_state: int = 42,
                      suggest_metrics: bool = True):
    print(f"\n=== Cenário {label} ===")
    gen = SimpleSyntheticDataGenerator(random_state=random_state)
    base_df = gen.generate(n_samples, generator_specs)
    inducer_local = SimpleDriftInducer(random_state=random_state+7)
    drifted_df, drift_meta = inducer_local.induce(base_df, drift_config)
    print(f"Drift aplicado em colunas: {[c for c in drift_config.keys()]}")

    analyzer = DatasetAnalyzer()
    if suggest_metrics:
        stats_report, metric_suggestions = analyzer.analyze_dataset(base_df, drifted_df, target_column=[], suggest_drift_metrics=True)
        applicable_report = metric_suggestions
    else:
        stats_report = analyzer.analyze_dataset(base_df, drifted_df, target_column=[], suggest_drift_metrics=False)
        # construir estrutura mínima para calculator
        detected_types = analyzer.detect_column_types(base_df)
        applicable_report = {'columns': {f: {'feature_type': t, 'applicable_metrics': ['psi','hellinger_distance']} for f,t in detected_types.items()}}

    calculator = DriftMetricsCalculator()
    metrics_results = calculator.calculate_metrics_from_report(base_df, drifted_df, applicable_report)

    # Integração com DriftReportGenerator
    report_gen = DriftReportGenerator()
    integrated_stats = report_gen.integrate_dataset_statistics(stats_report)
    interpreted = report_gen.integrate_drift_metrics(metrics_results)

    # Construir sumário simples de severidades (baseado em PSI se existir)
    feature_severities = {}
    for f, res in metrics_results.items():
        if f.startswith('_'): continue
        psi_val = res.get('psi', {}).get('psi_value') if isinstance(res.get('psi'), dict) else None
        if psi_val is None:
            sev = 'UNKNOWN'
        else:
            if psi_val < 0.1: sev = 'LOW'
            elif psi_val < 0.2: sev = 'MEDIUM'
            elif psi_val < 0.25: sev = 'HIGH'
            else: sev = 'CRITICAL'
        feature_severities[f] = {'psi': psi_val, 'severity': sev}

    # Simples agregação
    order = ['NONE','LOW','MEDIUM','HIGH','CRITICAL','UNKNOWN']
    def sev_rank(s):
        return order.index(s) if s in order else len(order)
    global_severity = max((v['severity'] for v in feature_severities.values()), key=sev_rank) if feature_severities else 'NONE'

    # Estratégia (mock simples usando AdaptationStrategyEngine se existir engine real)
    try:
        from drift_strategy_adapt import DriftSeverity as _DriftSeverityEnum, AdaptationContext, ContinuousMonitoringStrategy, ThresholdAdjustmentStrategy, FeatureRecalibrationStrategy, PartialRetrainingStrategy, FullRetrainingStrategy, EmergencyFallbackStrategy
        # map severidade
        map_sev = {
            'LOW': _DriftSeverityEnum.LOW,
            'MEDIUM': _DriftSeverityEnum.MEDIUM,
            'HIGH': _DriftSeverityEnum.HIGH,
            'CRITICAL': _DriftSeverityEnum.CRITICAL,
            'NONE': _DriftSeverityEnum.NEGLIGIBLE,
            'UNKNOWN': _DriftSeverityEnum.LOW
        }
        affected = [f for f,v in feature_severities.items() if v['severity'] not in ['LOW','NONE','UNKNOWN']]
        context = AdaptationContext(
            drift_score = np.mean([v['psi'] for v in feature_severities.values() if v['psi'] is not None]) if feature_severities else 0.0,
            drift_severity = map_sev.get(global_severity, _DriftSeverityEnum.LOW),
            affected_features = affected,
            performance_impact = 0.0,
            model_confidence = 0.9,
            business_criticality = 'medium',
            available_resources = {'retraining_data_size': len(drifted_df), 'full_retraining_approved': True}
        )
        strategies = [ContinuousMonitoringStrategy(), ThresholdAdjustmentStrategy(), FeatureRecalibrationStrategy(), PartialRetrainingStrategy(), FullRetrainingStrategy(), EmergencyFallbackStrategy()]
        # Selecionar heurística: primeira que pode aplicar com maior estimated impact
        applicable = [(s.estimate_impact(context), s) for s in strategies if s.can_apply(context)]
        if applicable:
            best = max(applicable, key=lambda x: x[0])[1]
            action = best.name
        else:
            action = 'NoStrategy'
    except Exception as e:
        action = f'AdaptationError: {e}'

    summary = {
        'scenario': label,
        'drift_meta': drift_meta,
        'global_severity': global_severity,
        'n_features': len(feature_severities),
        'n_metrics_total': metrics_results.get('_summary', {}).get('total_metrics_calculated'),
        'action_suggested': action,
        'feature_severities': feature_severities
    }
    return summary, {'stats_report': stats_report, 'metric_suggestions': applicable_report, 'metrics_results': metrics_results, 'interpreted': interpreted, 'integrated_stats': integrated_stats}

print("Função run_full_scenario pronta.")

Função run_full_scenario pronta.


In [11]:
# Exemplo rápido de uso
specs = [
    FeatureSpec("f_float0", SimpleDataType.FLOAT, dist="normal", params={"mean":2, "std":1.5}),
    FeatureSpec("f_float1", SimpleDataType.FLOAT, dist="normal"),
    FeatureSpec("f_float2", SimpleDataType.FLOAT, dist="poisson"),
    FeatureSpec("f_int0", SimpleDataType.INT, dist="poisson", params={"lam":8}),
    FeatureSpec("f_cat_num0", SimpleDataType.CAT_NUM, n_categories=5),
    FeatureSpec("f_cat_str0", SimpleDataType.CAT_STR, n_categories=4),
    FeatureSpec("f_cat_str1", SimpleDataType.CAT_STR, n_categories=5),
    FeatureSpec("f_ord", SimpleDataType.ORDINAL, n_categories=5, order=["A","B","C","D","E"]),
    FeatureSpec("f_bool", SimpleDataType.BOOLEAN, params={"probs":[0.7,0.3]}),
]

simple_gen = SimpleSyntheticDataGenerator(random_state=42)
reference_df = simple_gen.generate(1000, specs)


# Recriar demo após ajuste
inducer = SimpleDriftInducer(random_state=10)
print(reference_df.head())

   f_float0  f_float1  f_float2  f_int0 f_cat_num0 f_cat_str0 f_cat_str1  \
0  2.457076 -0.059283 -0.451951      11          1         C0         C1   
1  0.440024 -0.729287 -0.665878      12          1         C2         C1   
2  3.125677 -0.414473  0.434010       5          1         C0         C1   
3  3.410847  0.633910  0.251854      12          1         C2         C0   
4 -0.926553  0.002993 -1.404792       6          1         C2         C0   

  f_ord  f_bool  
0     B       0  
1     C       1  
2     E       0  
3     C       0  
4     B       0  


In [16]:
def run_full_test(reference_df,current_df, model=None):
    print("\n=== Executando análise completa de drift ===")
    analyzer = DatasetAnalyzer()

    dataset_analysis_report, metrics = analyzer.analyze_dataset(reference_df=reference_df,
                                                                current_df=current_df,
                                                                suggest_drift_metrics=True)


    # Instanciar calculadora inteligente
    metrics_calculator = DriftMetricsCalculator()

    # print(f"\nCALCULANDO MÉTRICAS...")
    # print("-" * 45)

    # Usar o relatório do SmartDriftAnalyzer para calcular apenas métricas aplicáveis
    metrics_results = metrics_calculator.calculate_metrics_from_report(
        reference_df=reference_df,
        current_df=current_df,
        analysis_report=metrics
    )
    print(metrics_results)
    report_generator = DriftReportGenerator()

    report = report_generator.generate_report(
        statistical_report=dataset_analysis_report,
        reference_df=reference_df,
        current_df=current_df,
        drift_metrics_results=metrics_results,
        include_model_impact=False  # Sem modelo para esta demonstração
    )

    report_generator.print_report(report)
    adaptation_engine = AdaptationStrategyEngine(auto_execution_threshold=0.7)
    
    # # Contexto de negócio para teste
    # business_context = {
    #     'criticality': 'high',
    #     'data_size': len(current_df),
    #     'auto_retrain': True,
    #     'model': LogisticRegression(),  # Modelo fictício para teste
    #     'time_limit': None,
    #     'available_resources': {
    #         'retraining_data_size': len(current_df),
    #         'full_retraining_approved': True,
    #         'model': LogisticRegression()
    #     }
    # }
    
    # # try:
    # # Executar adaptação com o relatório científico
    # adaptation_result = adaptation_engine.execute_adaptation_from_report(
    #     drift_report=report,
    #     business_context=business_context,
    #     auto_execute=False
    # )

    # persistence_manager = DriftPersistenceManager()

    # persistence_manager.register_execution(report, (adaptation_engine, adaptation_result.get('confidence', None), adaptation_result.get('decision')))

In [18]:
drift_config_demo = {
    "f_float0": {"type":"soft_shift"},
    # "f_int": {"type":"sudden_mean_shift", "severity":1.0},
    # "f_cat_str0": {"type":"category_frequency_shift", "severity":2.0},
    # "f_cat_num": {"type":"new_categories", "severity":2.0, "n_new":2},
    # "f_bool": {"type":"missingness", "severity":3.0},
    # "f_ord": {"type":"seasonal", "severity":1.0}  # ignorado por ser ordinal (não numérico direto)
}

current_df, drift_meta = inducer.induce(reference_df, drift_config_demo)

run_full_test(reference_df, current_df)


=== Executando análise completa de drift ===
{'f_float0': {'psi': {'psi_value': 0.004145208254483248, 'regulatory_compliant': True, 'bins_used': 12, 'method': 'doane_unified', 'binning_consistency': 'unified_with_other_metrics'}, 'kl_divergence': {'kl_divergence': 0.002088588885054758, 'bins_used': 12, 'method': 'doane_binning'}, 'js_divergence': {'js_divergence': 0.0005166304393342425, 'bins_used': 12, 'method': 'doane_binning'}, 'ks_test': {'ks_statistic': 0.029, 'p_value': 0.7946637387576738, 'method': 'cdf_based'}, 'hellinger_distance': {'hellinger_distance': 0.022746169161959963, 'bins_used': 12, 'method': 'doane_binning'}, 'wasserstein_distance': {'wasserstein_distance': 0.07415417665473804, 'normalized_distance': 0.007137615867185876, 'normalized_by_iqr': 0.03843580379844912, 'data_range': 10.389208110182954, 'reference_iqr': 1.9292994897047049}, '_metadata': {'feature_name': 'f_float0', 'total_applicable': 6, 'calculated': 6, 'skipped': 0, 'applicable_metrics': ['psi', 'kl_div

## **CALIBRAÇÃO DOS TIPOS DE DRIFT**

### **Problema Identificado:**
O tipo `soft_shift` original usava `severity × 0.5`, gerando D ≈ 0.10-0.15 (efeito **moderado**), não "soft" (trivial).

### **Solução Implementada:**
Novos tipos calibrados seguindo **Cohen (1988)** e **Sawilowsky (2009)**:

| Tipo | Multiplicador | Efeito Esperado (D) | Severidade Esperada | Uso Recomendado |
|------|---------------|---------------------|---------------------|-----------------|
| `trivial_shift` | × 0.1 | D < 0.05 | **LOW** | Flutuação normal, teste de sensibilidade |
| `small_shift` | × 0.25 | 0.05 ≤ D < 0.10 | **MEDIUM** | Mudança detectável mas operacionalmente pequena |
| `moderate_shift` | × 0.5 | 0.10 ≤ D < 0.20 | **HIGH** | Mudança substancial requerendo investigação |
| `large_shift` | × 1.0 | D ≥ 0.20 | **CRITICAL** | Mudança drástica, ação imediata |

### **Cálculo do Efeito:**
Para uma feature com std = 1.5 e severity = 1.0:
- `trivial_shift`: delta = 0.1 × 1.0 × 1.5 = **0.15** → D ≈ 0.03-0.04
- `small_shift`: delta = 0.25 × 1.0 × 1.5 = **0.375** → D ≈ 0.06-0.08
- `moderate_shift`: delta = 0.5 × 1.0 × 1.5 = **0.75** → D ≈ 0.11-0.13
- `large_shift`: delta = 1.0 × 1.0 × 1.5 = **1.5** → D ≈ 0.22-0.25

### **Mapeamento para Métricas:**
- **KS Test (D):** Máxima diferença entre CDFs cumulativas
- **Wasserstein (normalizado):** % do IQR deslocado
- **PSI:** Soma ponderada das diferenças log(p₁/p₂)

### **Legado (compatibilidade):**
- `soft_shift` → agora equivale a `trivial_shift` (emite warning)
- `hard_shift` → agora equivale a `large_shift × 1.5` (emite warning)

In [14]:
# 🧪 TESTE COMPARATIVO: Validação da Calibração dos Tipos de Drift
print("="*80)
print("VALIDAÇÃO DOS TIPOS DE DRIFT CALIBRADOS")
print("="*80)

drift_types_to_test = [
    ("trivial_shift", "LOW"),
    ("small_shift", "MEDIUM"),
    ("moderate_shift", "HIGH"),
    ("large_shift", "CRITICAL"),
    ("soft_shift", "LOW"),  # legacy
]

results_calibration = []

for drift_type, expected_severity in drift_types_to_test:
    print(f"\n{'─'*80}")
    print(f"📊 Testando: {drift_type} (esperado: {expected_severity})")
    print(f"{'─'*80}")
    
    # CORREÇÃO: Aplicar drift usando o método correto
    test_config = {"f_float0": {"type": drift_type, "severity": 1.0}}
    test_df, test_meta = inducer.induce(reference_df, test_config)
    
    # Debug: Verificar se drift foi aplicado
    delta_meta = test_meta.get('f_float0', {}).get('delta', 0)
    print(f"🔍 DEBUG: Delta dos metadados = {delta_meta:.4f}")
    
    # Calcular delta real comparando médias
    ref_mean = reference_df['f_float0'].mean()
    test_mean = test_df['f_float0'].mean()
    delta_real = test_mean - ref_mean
    ref_std = reference_df['f_float0'].std()
    delta_in_sigma = delta_real / ref_std if ref_std > 0 else 0
    
    print(f"🔍 DEBUG: Delta real (média) = {delta_real:.4f} ({delta_in_sigma:.4f}σ)")
    
    # Analisar
    analyzer_test = DatasetAnalyzer()
    stats_report, metrics_suggestions = analyzer_test.analyze_dataset(
        reference_df=reference_df,
        current_df=test_df,
        suggest_drift_metrics=True
    )
    
    # Calcular métricas
    calc_test = DriftMetricsCalculator()
    metrics_results = calc_test.calculate_metrics_from_report(
        reference_df=reference_df,
        current_df=test_df,
        analysis_report=metrics_suggestions
    )
    
    # Gerar relatório
    report_gen_test = DriftReportGenerator()
    drift_report = report_gen_test.integrate_drift_metrics(metrics_results)
    
    # Extrair valores (CORREÇÃO: acessar metric_interpretations)
    f_float0_results = drift_report.get('f_float0', {})
    metric_interpretations = f_float0_results.get('metric_interpretations', {})
    
    ks_severity = None
    wasserstein_severity = None
    ks_stat = None
    ks_p = None
    wass_dist = None
    wass_norm = None
    
    # Buscar KS Test
    if 'ks_test' in metric_interpretations:
        ks_data = metric_interpretations['ks_test']
        ks_severity = ks_data.get('severity')
        ks_stat = ks_data.get('raw_value')  # D statistic
    
    # Buscar Wasserstein
    if 'wasserstein_distance' in metric_interpretations:
        wass_data = metric_interpretations['wasserstein_distance']
        wasserstein_severity = wass_data.get('severity')
        wass_dist = wass_data.get('raw_value')
    
    # Buscar normalização
    if 'f_float0' in metrics_results:
        wass_data = metrics_results['f_float0'].get('wasserstein_distance', {})
        wass_norm = wass_data.get('normalized_by_iqr')
    
    # Metadata do drift
    expected_effect = test_meta.get('f_float0', {}).get('expected_effect', 'N/A')
    warning = test_meta.get('f_float0', {}).get('warning', '')
    
    # Consenso de severidade (pior entre KS e Wasserstein)
    severity_order = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
    consensus_severity = max(
        [s for s in [ks_severity, wasserstein_severity] if s in severity_order],
        key=lambda s: severity_order.index(s)
    ) if any(s in severity_order for s in [ks_severity, wasserstein_severity]) else 'N/A'
    
    match = '✅' if consensus_severity == expected_severity else '❌'
    
    # CORREÇÃO: Formatar valores ANTES de adicionar ao dicionário
    ks_stat_str = f"{ks_stat:.4f}" if (ks_stat is not None and isinstance(ks_stat, (int, float))) else "N/A"
    wass_norm_str = f"{wass_norm*100:.1f}%" if (wass_norm is not None and isinstance(wass_norm, (int, float))) else "N/A"
    
    results_calibration.append({
        'Tipo': drift_type,
        'Delta (σ)': f"{delta_in_sigma:.3f}",
        'KS_D': ks_stat_str,
        'KS_Sev': ks_severity or 'N/A',
        'Wass_%IQR': wass_norm_str,
        'Wass_Sev': wasserstein_severity or 'N/A',
        'Consenso': consensus_severity,
        'Esperado': expected_severity,
        'Match': match,
        'Efeito': expected_effect,
        'Warning': warning
    })
    
    print(f"  Delta aplicado: {delta_in_sigma:.3f}σ")
    
    print(f"  KS: D={ks_stat_str} → {ks_severity or 'N/A'}")
    print(f"  Wasserstein: {wass_norm_str} IQR → {wasserstein_severity or 'N/A'}")
    print(f"  Consenso: {consensus_severity} (esperado: {expected_severity}) {match}")
    if warning:
        print(f"  ⚠️ {warning}")

print(f"\n{'='*80}")
print("📋 RESUMO COMPARATIVO DA CALIBRAÇÃO:")
print(f"{'='*80}\n")
summary_calib_df = pd.DataFrame(results_calibration)
display(summary_calib_df)

# Análise de acurácia
matches = sum(1 for r in results_calibration if r['Match'] == '✅')
total = len(results_calibration)
accuracy = matches / total * 100

print(f"\n{'='*80}")
print(f"📊 ACURÁCIA DA CALIBRAÇÃO: {matches}/{total} ({accuracy:.1f}%)")
print(f"{'='*80}")

if accuracy >= 80:
    print("✅ Calibração APROVADA - Tipos de drift alinhados com severidades esperadas")
else:
    print("⚠️ Calibração requer ajuste - Revisar multiplicadores ou thresholds")
    print("\n🔍 TIPOS QUE FALHARAM:")
    for r in results_calibration:
        if r['Match'] == '❌':
            print(f"  • {r['Tipo']}: Esperado {r['Esperado']}, obtido {r['Consenso']} (Delta={r['Delta (σ)']})")


VALIDAÇÃO DOS TIPOS DE DRIFT CALIBRADOS

────────────────────────────────────────────────────────────────────────────────
📊 Testando: trivial_shift (esperado: LOW)
────────────────────────────────────────────────────────────────────────────────
🔍 DEBUG: Delta dos metadados = 0.1483
🔍 DEBUG: Delta real (média) = 0.0742 (0.0500σ)


  Delta aplicado: 0.050σ
  KS: D=N/A → LOW
  Wasserstein: 3.8% IQR → LOW
  Consenso: LOW (esperado: LOW) ✅

────────────────────────────────────────────────────────────────────────────────
📊 Testando: small_shift (esperado: MEDIUM)
────────────────────────────────────────────────────────────────────────────────
🔍 DEBUG: Delta dos metadados = 0.3708
🔍 DEBUG: Delta real (média) = 0.1854 (0.1249σ)
  Delta aplicado: 0.125σ
  KS: D=N/A → MEDIUM
  Wasserstein: 9.6% IQR → MEDIUM
  Consenso: MEDIUM (esperado: MEDIUM) ✅

────────────────────────────────────────────────────────────────────────────────
📊 Testando: moderate_shift (esperado: HIGH)
────────────────────────────────────────────────────────────────────────────────
🔍 DEBUG: Delta dos metadados = 0.7415
🔍 DEBUG: Delta real (média) = 0.3708 (0.2499σ)
  Delta aplicado: 0.250σ
  KS: D=N/A → HIGH
  Wasserstein: 19.2% IQR → HIGH
  Consenso: HIGH (esperado: HIGH) ✅

──────────────────────────────────────────────────────────────────────────────

,Tipo,Delta (σ),KS_D,KS_Sev,Wass_%IQR,Wass_Sev,Consenso,Esperado,Match,Efeito,Warning
0,trivial_shift,0.050,N/A,LOW,3.8%,LOW,LOW,LOW,✅,trivial (D < 0.05),
1,small_shift,0.125,N/A,MEDIUM,9.6%,MEDIUM,MEDIUM,MEDIUM,✅,small (0.05 ≤ D < 0.10),
2,moderate_shift,0.250,N/A,HIGH,19.2%,HIGH,HIGH,HIGH,✅,moderate (0.10 ≤ D < 0.20),
3,large_shift,0.500,N/A,CRITICAL,38.4%,CRITICAL,CRITICAL,CRITICAL,✅,large (D ≥ 0.20),
4,soft_shift,0.050,N/A,LOW,3.8%,LOW,LOW,LOW,✅,N/A,soft_shift deprecated: use trivial_shift instead



📊 ACURÁCIA DA CALIBRAÇÃO: 5/5 (100.0%)
✅ Calibração APROVADA - Tipos de drift alinhados com severidades esperadas
